# 第17章　モダリティ別・前処理レシピ集**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## 17.1　CT ― HU値とウィンドウ処理

In [ ]:
def ct_window(hu, center, width):    low, high = center - width/2, center + width/2    x = np.clip(hu, low, high)    return (x - low) / (high - low)          # 0〜1へ# 代表的なウィンドウ（目的の臓器で使い分ける）abdomen = ct_window(hu, center=40,  width=400)   # 腹部軟部（肝・膵）lung    = ct_window(hu, center=-600, width=1500)  # 肺野bone    = ct_window(hu, center=400,  width=1800)  # 骨（施設・目的で幅がある。中心300〜600/幅1500〜2800程度）brain   = ct_window(hu, center=40,  width=80)     # 脳

## 実データのDICOMでよくある前処理の分岐

In [ ]:
import numpy as np, pydicomdef load_dicom_hu(path):    ds = pydicom.dcmread(path)    arr = ds.pixel_array.astype(np.float32)    slope = float(getattr(ds, "RescaleSlope", 1.0))       # タグ欠落に既定値    inter = float(getattr(ds, "RescaleIntercept", 0.0))    # MONOCHROME1（値が大きいほど黒）は、HUに直す“前”に、格納値の段階で反転させる。    # HU変換後に arr.max() - arr とやると、返る値はもうHUではなくなり、後段のウィンドウ処理が    # すべて意味を失う。また arr.max() はスライスごとに変わるので、ボリューム内で物差しがずれる。    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":        vmax = 2 ** int(getattr(ds, "BitsStored", 16)) - 1  # 理論上の最大値（データ依存にしない）        arr = vmax - arr    arr = arr * slope + inter                              # 生画素→HU（第7章）    return arr, ds

In [ ]:
pairs = [(p, pydicom.dcmread(p)) for p in paths]pairs.sort(key=lambda t: float(t[1].ImagePositionPatient[2]))  # z座標で並べる（名前でなく。アキシャル収集のCTが前提）# 生の格納画素値のまま積むとHUにならない。必ず上の load_dicom_hu を通すvolume = np.stack([load_dicom_hu(p)[0] for p, _ in pairs])   # 正しい頭尾方向のHUボリューム

## 17.2　MRI ― 相対値のzスコア正規化

In [ ]:
def mri_zscore(vol, mask=None):    region = vol[mask > 0] if mask is not None else vol[vol > 0]  # 背景を除く    mean, std = region.mean(), region.std()    return (vol - mean) / (std + 1e-8)

## 17.3　眼底写真 ― CLAHEと余白の切り落とし

In [ ]:
import cv2, numpy as npdef fundus_prep(bgr):    # ① 眼底の円形領域に外接する矩形で切り出す（周囲の黒い余白は情報を持たず、位置ずれの原因になる）    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)    mask = gray > 10                                    # 黒背景と眼底を分ける    ys, xs = np.where(mask)    bgr = bgr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # 外接矩形で切り出す    # ② 明度チャネルにCLAHE（局所コントラストを整える）    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))    lab[..., 0] = clahe.apply(lab[..., 0])    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

## 17.4　X線 ― 正規化と、アスペクト比

In [ ]:
import numpy as npfrom pydicom.pixels import apply_modality_lut, apply_voi_lut# ① 正攻法：装置が意図した表示範囲（VOI LUT）を使うarr = apply_modality_lut(ds.pixel_array, ds)if "WindowCenter" in ds:                                  # 入っていれば、これを尊重する    arr = apply_voi_lut(arr, ds)    xr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)else:    # ② VOI LUTが無いとき：パーセンタイルでクリップしてから伸縮（外れ値1画素に壊されない）    lo, hi = np.percentile(arr, [0.5, 99.5])    xr = np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)# ③ 最後の手段：素朴な min-max（焼き込み文字や飽和画素があると崩れる）# xr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)   # 0〜1へ# 拡張は回転・平行移動・明るさなど、臨床的に許されるものに限る